# 03 · Desarrollo de modelos multimodales

Entrenamiento y comparación de 5 arquitecturas multimodales (ResNet50/Xception + TabNet/MLP) con datasets `tf.data` de alto rendimiento, fine-tuning progresivo y selección del mejor modelo por F1 macro.

In [ ]:
# --- Configuración del entorno (portable: local / Colab) ---
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Project-ComputerVision')
    IS_COLAB = True
except ImportError:
    PROJECT_ROOT = Path().resolve().parent.parent
    IS_COLAB = False

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import CFG  # noqa: E402

print(f"Entorno: {'Colab' if IS_COLAB else 'Local'}")
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Modelo ganador: {CFG.BEST_MODEL_KERAS.name}")

In [ ]:
# --- Librerías Estándar de Python ---
import os
import warnings
from pathlib import Path

# --- Manejo de Datos y Visualización ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Machine Learning (Scikit-Learn) ---
from sklearn.preprocessing import LabelEncoder

# --- Deep Learning (TensorFlow y Keras) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.layers import (Activation, BatchNormalization, Conv2D, Input, add)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# --- Modelos Especializados ---
from tabnet_keras import TabNetClassifier
from tensorflow.keras.applications import ResNet50, ResNet50V2


warnings.filterwarnings('ignore')

In [ ]:
%pip install -q tabnet-keras

In [ ]:
from functions.multimodal_config import multimodal_generator
from functions.utilidades import get_run_logdir
from functions.xception_model import entry_flow, middle_flow, exit_flow

In [ ]:
filename = str(PROJECT_ROOT + '/preprocessed_multimodal/preprocessed_multimodal_data_brasil.pkl')
loaded_data = pd.read_pickle(filename)

# filename = 'preprocessed_multimodal/preprocessed_multimodal_data_brasil.pkl'
# loaded_data = pd.read_pickle(path / filename)

df_train = loaded_data['df_train']
df_train_balanced = loaded_data['df_train_balanced']

df_val = loaded_data['df_val']
df_test = loaded_data['df_test']

X_train_processed = loaded_data['X_train_processed']
# X_train_processed_balanced = loaded_data['X_train_balanced'],
X_val_processed = loaded_data['X_val_processed']
X_test_processed = loaded_data['X_test_processed']

y_train = loaded_data['y_train']
y_train_balanced = loaded_data['y_train_balanced']

y_val = loaded_data['y_val']
y_test = loaded_data['y_test']

preprocessor = loaded_data['preprocessor']

print("¡Datos cargados y listos para el modelado!")

In [ ]:
print(f"Tamaño del conjunto de entrenamiento: {len(df_train)}")
print(f"Tamaño del conjunto de validación: {len(df_val)}")
print(f"Tamaño del conjunto de prueba: {len(df_test)}")

In [ ]:
print(f"Tamaño del conjunto de entrenamiento balanceado: {len(df_train_balanced)}")

Cargando las imagenes a Colab:

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
# y_train_encoded_balanced = label_encoder.fit_transform(y_train_balanced)

y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Ejemplo de etiqueta original:", y_train.iloc[0])
print("Después de LabelEncoder:", y_train_encoded[0])
print("Clases encontradas por el encoder:", label_encoder.classes_)
num_classes = len(label_encoder.classes_)

y_train_cat = to_categorical(y_train_encoded, num_classes=num_classes)
y_val_cat = to_categorical(y_val_encoded, num_classes=num_classes)
y_test_cat = to_categorical(y_test_encoded, num_classes=num_classes)

print("\nDespués de to_categorical:", y_train_cat[0])

In [ ]:
# # Esto ha sido para entrenar el modelo de prueba
# df_train_ = df_train.head(6)
# df_val_ = df_val.head(6)

# X_train_processed_ = X_train_processed[:6, ]
# X_val_processed_ = X_val_processed[:6, ]

# y_train_cat_ = y_train_cat[:6]
# y_val_cat_ = y_val_cat[:6]

In [ ]:
df_train_ = df_train
df_val_ = df_val

X_train_processed_ = X_train_processed
X_val_processed_ = X_val_processed

y_train_cat_ = y_train_cat
y_val_cat_ = y_val_cat

***

In [ ]:
BATCH_SIZE = 64
IMG_SHAPE = (256, 256, 3)

In [ ]:
source_path = str(PROJECT_ROOT + '/Data/zr7vgbcyr2-1/PAD_UFES_20_ALL_IMAGES')

local_img_path = '/tmp/local_images/'

print("Copiando imágenes al disco local para acelerar el entrenamiento...")
os.makedirs(local_img_path, exist_ok=True)
!rsync -a --info=progress2 {source_path}/ {local_img_path}
print("Copia de imágenes completada.")

In [ ]:
data_augmentation_pipeline = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.042),      # Corresponde a ~15 grados
    layers.RandomTranslation(height_factor=0.15, width_factor=0.15),
    layers.RandomZoom(height_factor=0.15),
    layers.RandomBrightness(factor=0.2)        # Corresponde a un brightness_range de [0.8, 1.2]
], name="gpu_augmentation")

In [ ]:
# local_img_path = r'G:\Mi unidad\Project-ComputerVision\Data\zr7vgbcyr2-1\PAD_UFES_20_ALL_IMAGES'

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
def load_and_resize_image(image_path, tabular_row, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(image, channels=3)
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, [IMG_SHAPE[0], IMG_SHAPE[1]])
    image = tf.cast(image, tf.float32)

    return (image, tabular_row), label

def create_fast_dataset(image_paths, tabular_data, labels, augment=False):

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, tabular_data, labels))
    dataset = dataset.map(load_and_resize_image, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.map(lambda x, y: ((data_augmentation_pipeline(x[0]), x[1]), y),
                              num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.map(lambda x, y: ((preprocess_input(x[0]), x[1]), y),
                          num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.shuffle(buffer_size=1024)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

***
Verificación que las imagenes realmente apunten al objetivo deseado.

In [ ]:
def deprocess_image(img):
    """Invierte el preprocesamiento de VGG16 para visualización."""
    img = img.copy()

    img[..., 0] += 103.939
    img[..., 1] += 116.779
    img[..., 2] += 123.68
    # Invierte de BGR a RGB
    img = img[..., ::-1]
    # Recorta a rango válido [0, 255] y convierte a entero
    img = np.clip(img, 0, 255).astype('uint8')
    return img

In [ ]:
# Creando el nuevo path para buscar las imágenes
train_image_paths_local = [os.path.join(local_img_path, fname) for fname in df_train['img_id']]
val_image_paths_local = [os.path.join(local_img_path, fname) for fname in df_val['img_id']]

In [ ]:
print("Creando dataset de verificación...")
verify_dataset = create_fast_dataset(
    train_image_paths_local,
    X_train_processed,
    y_train_cat,
    augment=False
)

In [ ]:
(images_batch, tabular_batch), labels_batch = next(iter(verify_dataset))

In [ ]:
idx_to_check = 0

print(f"\n--- 1. DATOS DEL PIPELINE (Índice {idx_to_check}) ---")

# Obtiene los datos del pipeline
img_from_ds = images_batch[idx_to_check]
tab_from_ds = tabular_batch[idx_to_check]
lbl_from_ds = labels_batch[idx_to_check]
lbl_index_ds = np.argmax(lbl_from_ds)

print(f"Etiqueta (índice): {lbl_index_ds}")
print(f"Datos Tabulares:\n{tab_from_ds}")

In [ ]:
plt.imshow(deprocess_image(img_from_ds.numpy()))
plt.title(f"Imagen del Pipeline (Etiqueta: {lbl_index_ds})")
plt.axis('off')
plt.show()

print(f"\n--- 2. DATOS ORIGINALES (Índice {idx_to_check}) ---")

In [ ]:
original_path = train_image_paths_local[idx_to_check]
original_tab = X_train_processed[idx_to_check]
original_lbl = y_train_cat[idx_to_check] # También es one-hot
original_lbl_index = np.argmax(original_lbl)

print(f"Ruta de Imagen: {original_path}")
print(f"Etiqueta (índice): {original_lbl_index}")
print(f"Datos Tabulares:\n{original_tab}")

print("Revisa que la imagen de arriba coincida con la ruta impresa.")
print(f"¿Coinciden los índices de etiqueta? {'Sí' if lbl_index_ds == original_lbl_index else 'No'}")

In [ ]:
testing_with_cloud = df_train[df_train.img_id == "PAT_924_1757_784.png"].iloc[[0], ]

In [ ]:
a1 = preprocessor.transform(testing_with_cloud)

Esto se utiliza para una revisin manual de las coicidencias exactas.

In [ ]:
a1

***

In [ ]:
len_train = len(df_train_)
len_val = len(df_val_)

train_dataset = create_fast_dataset(
    train_image_paths_local,
    X_train_processed,
    y_train_cat,
    augment=True
)

val_dataset = create_fast_dataset(
    val_image_paths_local,
    X_val_processed,
    y_val_cat,
    augment=False
)

### Modelo multimodal 1, probando la arquitectura con ResNetV250

In [ ]:
img_input = keras.layers.Input(shape=IMG_SHAPE, name="img_input")
# Apply CenterCrop here to match ResNet50 input shape
vision_branch = layers.CenterCrop(height=224, width=224)(img_input)

base_model = ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Congela la base para no re-entrenarla
base_model.trainable = False

In [ ]:
# Pass the cropped image input to the base_model
vision_branch = base_model(vision_branch, training=False)
vision_branch = layers.GlobalAveragePooling2D()(vision_branch)

Como he realizado `oversampling` entonces quito los weights.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = label_encoder.classes_

# Calcular pesos con esas etiquetas
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Crear diccionario usando los índices que el LabelEncoder asignó
class_weights_dict = {
    idx: weight for idx, weight in enumerate(class_weights)
}

print("Pesos de clase:", class_weights_dict)

In [ ]:
TABULAR_SHAPE = (X_train_processed.shape[1],)

In [ ]:
table_input = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input")

In [ ]:
tabnet = TabNetClassifier(n_classes= 6)
tabular_branch = tabnet.tabnet_encoder(table_input)

In [ ]:
from tensorflow.keras import regularizers

In [ ]:
combined = layers.concatenate([vision_branch, tabular_branch])

# Cabezal de clasificación
head = layers.Dense(256, activation="relu")(combined)
head = layers.Dropout(0.4)(head)
output = layers.Dense(6, activation="softmax", name="output")(head)

In [ ]:
model = keras.models.Model(inputs=[img_input, table_input], outputs=[output])

model.summary()

In [ ]:
run_logdir = get_run_logdir(str(PROJECT_ROOT + "/my_logs"))

In [ ]:
checkpoint_cb = keras.callbacks.ModelCheckpoint(str(PROJECT_ROOT + "/models/model_RestNet50_1.keras"),
                                                save_best_only= True,
                                                monitor="val_f1_score_macro",
                                                mode="max")
early_stopping = keras.callbacks.EarlyStopping(monitor="val_f1_score_macro",
                                               patience=10,
                                               restore_best_weights=True,
                                               mode="max")
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

In [ ]:
f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')
model.load_weights(str(PROJECT_ROOT + "/models/model_RestNet50_1.keras"))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

In [ ]:
history = model.fit(
    train_dataset,
    epochs= 70,
    validation_data=val_dataset,
    steps_per_epoch= len_train // BATCH_SIZE,
    validation_steps= len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)

In [ ]:
history_ = pd.DataFrame(history.history)

In [ ]:
history_.plot()
plt.show()

In [ ]:
history_fine_tune = model.fit(
    train_dataset,
    epochs=60,
    initial_epoch=history.epoch[-1], # Continúa desde donde se quedó
    validation_data=val_dataset,
    steps_per_epoch= len_train // BATCH_SIZE,
    validation_steps= len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)

Mejorando el modelo - Parte 2

In [ ]:
print(f"Cargando el mejor modelo estable (71.8% F1)...")
model.load_weights(str(PROJECT_ROOT + "/models/model_RestNet50_1.keras"))
print("Pesos cargados con éxito.")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

print("--- AJUSTE 1: Continuando entrenamiento con LR reducido... ---")

last_epoch = history.epoch[-1]

history_decay = model.fit(
    train_dataset,
    epochs=70,
    initial_epoch=last_epoch,
    validation_data=val_dataset,
    steps_per_epoch= len_train // BATCH_SIZE,
    validation_steps= len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)
print("--- AJUSTE 1 Completado. ---")

# Último fine-Tiunning

In [ ]:
CHECKPOINT_PATH = str(PROJECT_ROOT + "/models/model_RestNet50_1.keras")

In [ ]:
img_input = keras.layers.Input(shape=IMG_SHAPE, name="img_input")
table_input = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input")
x = layers.CenterCrop(height=224, width=224)(img_input)

base_model = ResNet50(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False
x = base_model(x, training=False)
vision_branch = layers.GlobalAveragePooling2D()(x)

tabnet = TabNetClassifier(n_classes=6)
tabular_branch = tabnet.tabnet_encoder(table_input)

combined = layers.concatenate([vision_branch, tabular_branch])
head = layers.Dense(256, activation="relu")(combined)
head = layers.Dropout(0.4)(head)
output = layers.Dense(6, activation="softmax", name="output")(head)

model = keras.models.Model(inputs=[img_input, table_input], outputs=[output])

print(f"Cargando el mejor modelo estable (71.8% F1) desde {CHECKPOINT_PATH}...")
model.load_weights(CHECKPOINT_PATH)
print("Pesos cargados.")

In [ ]:
print("Descongelando las últimas capas del modelo base...")
base_model.trainable = True

total_layers = len(base_model.layers)
layers_to_freeze = total_layers - 15

print(f"Total de capas en ResNet50: {total_layers}. Congelando las primeras {layers_to_freeze} capas.")

for i, layer in enumerate(base_model.layers):
    if i < layers_to_freeze:
        layer.trainable = False
    else:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

print(f"Capas entrenables en base_model: {len(base_model.trainable_weights)}")

In [ ]:
f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

print("--- AJUSTE FINAL: Iniciando Fine-Tuning... ---")

last_epoch = history.epoch[-1]

history_fine_tune = model.fit(
    train_dataset,
    epochs=last_epoch + 20,
    initial_epoch=last_epoch,
    validation_data=val_dataset,
    steps_per_epoch= len_train // BATCH_SIZE,
    validation_steps= len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)
print("--- AJUSTE FINAL Completado. ---")

***

In [ ]:
tf.keras.backend.clear_session(
    free_memory=True
)

### Modelo multimodal 1, probando la arquitectura con modelo básico

Desarrollando la entrada de imagen:

In [ ]:
IMG_SHAPE = (256, 256, 3)

In [ ]:
img_input_basic = keras.layers.Input(shape=IMG_SHAPE, name="img_input_basic")

vision_branch_basic = layers.Conv2D(32, (3, 3), activation='relu')(img_input_basic)
vision_branch_basic = layers.MaxPooling2D((2, 2))(vision_branch_basic)
vision_branch_basic = layers.Conv2D(64, (3, 3), activation='relu')(vision_branch_basic)
vision_branch_basic = layers.MaxPooling2D((2, 2))(vision_branch_basic)
vision_branch_basic = layers.Conv2D(128, (3, 3), activation='relu')(vision_branch_basic)
vision_branch_basic = layers.MaxPooling2D((2, 2))(vision_branch_basic)
vision_branch_basic = layers.Flatten()(vision_branch_basic)
vision_branch_basic = layers.Dense(128, activation='relu')(vision_branch_basic)


In [ ]:
vision_branch_basic

Desarrollo del modelo para el texto a nivel básico

In [ ]:
TABULAR_SHAPE = (29, )

In [ ]:
table_input_basic = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input_basic")
tabular_branch_basic = layers.Dense(128, activation="relu")(table_input_basic)
tabular_branch_basic = layers.Dropout(0.3)(tabular_branch_basic)
tabular_branch_basic = layers.Dense(64, activation="relu")(tabular_branch_basic)
tabular_branch_basic = layers.Dropout(0.2)(tabular_branch_basic)
tabular_branch_basic = layers.Dense(64, activation="relu")(tabular_branch_basic)
abular_branch_basic = layers.Dropout(0.2)(tabular_branch_basic)
tabular_branch_basic = layers.Dense(32, activation="relu")(tabular_branch_basic)
tabular_branch_basic = layers.Dense(15, activation="relu")(tabular_branch_basic)

In [ ]:
combined_basic = layers.concatenate([vision_branch_basic, tabular_branch_basic])

# Cabezal de clasificación
head_basic = layers.Dense(64, activation="relu")(combined_basic)
head_basic = layers.Dense(32, activation="relu")(combined_basic)
output_basic = layers.Dense(6, activation="softmax", name="output")(head_basic)

In [ ]:
model = keras.models.Model(inputs=[img_input_basic, table_input_basic], outputs=[output_basic])

model.summary()

In [ ]:
try:
    model = model
    print("Modelo Xception cargado.")

    keras.utils.plot_model(
        model,
        # 1. Cambiamos de nuevo a .png
        to_file='arquitectura_basico.png',
        show_shapes=True,
        show_layer_names=True,
        show_layer_activations=True,
        expand_nested=True,
        
        dpi=300, 
        
        rankdir='LR' 
    )
    
    print("\n¡Diagrama guardado como 'xception_architecture_horizontal.png' (horizontal y alta resolución)!")

except ImportError:
    print("\nError: Faltan 'pydot' o 'graphviz'.")
    print("Asegúrate de que Graphviz esté instalado y en tu PATH.")
except Exception as e:
    print(f"\nOcurrió un error: {e}")

In [ ]:
run_logdir = get_run_logdir(str(PROJECT_ROOT + "/my_logs"))

In [ ]:
checkpoint_cb = keras.callbacks.ModelCheckpoint(str(PROJECT_ROOT + "/models/model_basic.keras"),
                                                save_best_only= True,
                                                monitor="val_f1_score_macro",
                                                mode="max")
early_stopping = keras.callbacks.EarlyStopping(patience= 20,
                                               restore_best_weights=True,
                                               monitor="val_f1_score_macro",
                                                mode="max"
                                               )
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

In [ ]:
CHECKPOINT_PATH = str(PROJECT_ROOT + "/models/model_basic.keras")

In [ ]:
f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')
model.load_weights(CHECKPOINT_PATH)
model.compile(
    optimizer = keras.optimizers.Nadam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

In [ ]:
history = model.fit(
    train_dataset,
    epochs=150,
    validation_data=val_dataset,
    steps_per_epoch= len_train // BATCH_SIZE,
    validation_steps=len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict,
    initial_epoch = history.epoch[-1]
)

In [ ]:
history_ = pd.DataFrame(history.history)

In [ ]:
history_.plot()
plt.grid()
plt.show()

# Modelo de XCEPTION - Tabular XCEPTION

In [ ]:
img_xception_input = keras.layers.Input(shape= IMG_SHAPE, name= "img_xception_input")

x = entry_flow(img_xception_input)
x = middle_flow(x)
vision_branch = exit_flow(x)

Modelo 1

In [ ]:
table_xception_input = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input_xception")

# --- RAMA 1 ---
stem = layers.Dense(32, activation="relu", kernel_initializer="he_normal")(table_xception_input)
stem = layers.Dense(64, activation="relu", kernel_initializer="he_normal")(stem)

branch_1 = layers.Dense(128, activation="relu", kernel_initializer="he_normal")(stem)
branch_1 = layers.Dense(128, activation="relu", kernel_initializer="he_normal")(branch_1)

# --- RAMA 2 (Ancha y con Atajo) ---
shortcut = layers.Dense(256, activation="relu", kernel_initializer="he_normal")(stem)
branch_2 = layers.Dropout(0.3)(shortcut)
branch_2 = layers.Dense(256, activation="relu", kernel_initializer="he_normal")(branch_2)

branch_2 = layers.add([shortcut, branch_2])

# --- RAMA 3 ---
# Una rama más simple que mira los datos originales desde otra perspectiva.
branch_3 = layers.Dense(128, activation="relu", kernel_initializer="he_normal")(table_xception_input)
branch_3 = layers.Dropout(0.4)(branch_3)

# --- FUSIÓN DE RAMAS (CONCATENACIÓN) ---
combined_features = layers.concatenate([
    branch_1,
    branch_2,
    branch_3
])

# --- CABEZAL FINAL (Head) ---
head1 = layers.Dense(728, activation="relu", kernel_initializer="he_normal")(combined_features)
head1 = layers.Dropout(0.5)(head1)
head1 = layers.Dense(728, activation="relu", kernel_initializer="he_normal")(head1)

Modelo 2

In [ ]:
def build_repeated_xception_block(input_tensor, repetitions=8):
    """
    Construye un bloque de capas estilo Xception que se repite N veces,
    utilizando conexiones residuales para estabilizar el entrenamiento.

    Args:
        input_tensor: El tensor de entrada al primer bloque.
        repetitions (int): El número de veces que se repetirá el bloque.

    Returns:
        El tensor de salida después de pasar por todos los bloques repetidos.
    """
    # Inicializamos la variable que se actualizará en cada iteración del bucle
    x = input_tensor

    # Bucle que repite el bloque de capas 8 veces
    for _ in range(repetitions):
        # 1. Guarda la entrada del bloque actual para la conexión residual
        residual = x

        x = layers.Dense(728, kernel_initializer="he_normal")(x)
        x = layers.BatchNormalization()(x)
        x = Activation("relu")(x)
        x = layers.Dropout(0.2)(x)

        x = layers.Dense(728, kernel_initializer="he_normal")(x)
        x = layers.BatchNormalization()(x)
        x = Activation("relu")(x)

        x = layers.add([x, residual])

    return x

In [ ]:
head2 = build_repeated_xception_block(head1, repetitions=8)

Modelo 3

In [ ]:
table_xception_branch_final = layers.Dense(728, kernel_initializer="he_normal")(head2)
table_xception_branch_final = layers.BatchNormalization()(table_xception_branch_final)
table_xception_branch_final = Activation("relu")(table_xception_branch_final)

table_xception_branch_final = layers.Dense(1024, kernel_initializer="he_normal")(table_xception_branch_final)
table_xception_branch_final = layers.BatchNormalization()(table_xception_branch_final)
table_xception_branch_final = Activation("relu")(table_xception_branch_final)

# --- CORRECCIÓN 3: Conexión Residual Significativa ---
if head2.shape[-1] != table_xception_branch_final.shape[-1]:
    head2_resized = layers.Dense(1024)(head2) # Ajusta la forma del atajo
    residual_connection_final = layers.add([table_xception_branch_final, head2_resized])
else:
    residual_connection_final = layers.add([table_xception_branch_final, head2])

Combinación de ambas partes

In [ ]:
combined_features = layers.concatenate([vision_branch, residual_connection_final])

Cabezal para la clasificación:

In [ ]:
final_head = layers.Dense(512, activation="relu")(combined_features)
final_head = layers.Dropout(0.5)(final_head)
final_output = layers.Dense(6, activation="softmax", name="final_output")(final_head)

In [ ]:
final_model = keras.models.Model(
    inputs= [img_xception_input, table_xception_input],
    outputs= [final_output]
)

In [ ]:
final_model.summary()

In [ ]:
f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')

final_model.compile(
    optimizer = keras.optimizers.Nadam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

In [ ]:
run_logdir = get_run_logdir(str(PROJECT_ROOT + "/my_logs"))

In [ ]:
CHECKPOINT_PATH = str(PROJECT_ROOT + "/models/model_advance_v1.keras")
checkpoint_cb = keras.callbacks.ModelCheckpoint(str(PROJECT_ROOT + "/models/model_advance_v1.keras"),
                                                save_best_only= True,
                                                monitor="val_f1_score_macro",
                                                mode="max"
                                                )
early_stopping = keras.callbacks.EarlyStopping(patience= 20,
                                               restore_best_weights=True,
                                               monitor="val_f1_score_macro",
                                                mode="max")
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

In [ ]:
history = final_model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    steps_per_epoch=len_train // BATCH_SIZE,
    validation_steps=len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)

In [ ]:
history_ = pd.DataFrame(history.history)

In [ ]:
history_.plot()
plt.grid()
plt.show()

***
El modelo entranado es demasiado fuerte para el conjunto de datos obtenidos, por tal razón, se baja la complejidad del modelo a uno más simple, en este caso se toma el modelo que se propuso en el ejercicio del modelo básico.
***

In [ ]:
from tensorflow.keras.applications import Xception

In [ ]:
from tensorflow.keras.applications.xception import preprocess_input

In [ ]:
def load_and_resize_image(image_path, tabular_row, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(image, channels=3)
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, [IMG_SHAPE[0], IMG_SHAPE[1]])
    image = tf.cast(image, tf.float32)

    return (image, tabular_row), label

def create_fast_dataset(image_paths, tabular_data, labels, augment=False):

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, tabular_data, labels))
    dataset = dataset.map(load_and_resize_image, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.map(lambda x, y: ((data_augmentation_pipeline(x[0]), x[1]), y),
                              num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.map(lambda x, y: ((preprocess_input(x[0]), x[1]), y),
                          num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.shuffle(buffer_size=1024)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

In [ ]:
img_input = keras.layers.Input(shape=IMG_SHAPE, name="img_input_xception") # (256, 256, 3)

# Recorta a (224, 224)
x = layers.CenterCrop(height=224, width=224)(img_input)

# Carga Xception con pesos 'imagenet' y sin el cabezal
base_model_xception = Xception(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Congela la base
base_model_xception.trainable = False

x = base_model_xception(x, training=False)
vision_branch = layers.GlobalAveragePooling2D()(x)

# --- 2. RAMA TABULAR (SIMPLE) ---
table_input = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input_xception")

tabular_branch_basic = layers.Dense(128, activation="relu")(table_input)
tabular_branch_basic = layers.Dropout(0.3)(tabular_branch_basic)
tabular_branch_basic = layers.Dense(64, activation="relu")(tabular_branch_basic)
tabular_branch_basic = layers.Dropout(0.2)(tabular_branch_basic)
tabular_branch_basic = layers.Dense(64, activation="relu")(tabular_branch_basic)

tabular_branch_basic = layers.Dropout(0.2)(tabular_branch_basic)

tabular_branch_basic = layers.Dense(32, activation="relu")(tabular_branch_basic)
tabular_branch_basic = layers.Dense(15, activation="relu")(tabular_branch_basic)

combined = layers.concatenate([vision_branch, tabular_branch_basic])

# Cabezal de clasificación simple
head = layers.Dense(128, activation="relu")(combined)
head = layers.Dropout(0.5)(head)
output = layers.Dense(6, activation="softmax", name="final_output")(head)

# --- 4. CONSTRUIR Y COMPILAR (CON LR CORRECTO) ---
high_model = keras.models.Model(
    inputs=[img_input, table_input],
    outputs=[output]
)

f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')

high_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

high_model.summary()

In [ ]:
run_logdir = get_run_logdir(str(PROJECT_ROOT + "/my_logs"))

In [ ]:
CHECKPOINT_PATH = str(PROJECT_ROOT + "/models/model_advance_1.keras")
checkpoint_cb = keras.callbacks.ModelCheckpoint(str(PROJECT_ROOT + "/models/model_advance_v1.keras"),
                                                save_best_only= True,
                                                monitor="val_f1_score_macro",
                                                mode="max"
                                                )
early_stopping = keras.callbacks.EarlyStopping(patience= 20,
                                               restore_best_weights=True,
                                               monitor="val_f1_score_macro",
                                                mode="max")
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

In [ ]:
history_high_1 = high_model.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    steps_per_epoch=len_train // BATCH_SIZE,
    validation_steps=len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)

***
Opción 2 con TabNet

In [ ]:
img_input = keras.layers.Input(shape=IMG_SHAPE, name="img_input_xception") # (256, 256, 3)

x = layers.CenterCrop(height=224, width=224)(img_input)

base_model_xception = Xception(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model_xception.trainable = False

x = base_model_xception(x, training=False)
vision_branch = layers.GlobalAveragePooling2D()(x)

table_input = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input_tabnet")

tabnet = TabNetClassifier(n_classes=6)
tabular_branch = tabnet.tabnet_encoder(table_input)

combined = layers.concatenate([vision_branch, tabular_branch])

head = layers.Dense(256, activation="relu")(combined)
head = layers.Dropout(0.4)(head)
output = layers.Dense(6, activation="softmax", name="final_output")(head)

high_model = keras.models.Model(
    inputs=[img_input, table_input],
    outputs=[output]
)

f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')

high_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

high_model.summary()

In [ ]:
run_logdir = get_run_logdir(str(PROJECT_ROOT + "/my_logs"))

In [ ]:
checkpoint_cb = keras.callbacks.ModelCheckpoint(str(PROJECT_ROOT + "/models/model_advance_Xception_TabNet.keras"),
                                                save_best_only= True,
                                                monitor="val_f1_score_macro",
                                                mode="max"
                                                )
early_stopping = keras.callbacks.EarlyStopping(patience= 20,
                                               restore_best_weights=True,
                                               monitor="val_f1_score_macro",
                                                mode="max")
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

In [ ]:
history_high_2 = high_model.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    steps_per_epoch=len_train // BATCH_SIZE,
    validation_steps=len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)

El modelo ganador en este caso ha sido ResNet50 + TabNet, por tanto, como Fine-Tunning se va a implementar las nuevas versiones de ResNet.

In [ ]:
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from tensorflow.keras.applications import ResNet50V2

In [ ]:
def load_and_resize_image(image_path, tabular_row, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(image, channels=3)
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, [IMG_SHAPE[0], IMG_SHAPE[1]])
    image = tf.cast(image, tf.float32)

    return (image, tabular_row), label

def create_fast_dataset(image_paths, tabular_data, labels, augment=False):

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, tabular_data, labels))
    dataset = dataset.map(load_and_resize_image, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.map(lambda x, y: ((data_augmentation_pipeline(x[0]), x[1]), y),
                              num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.map(lambda x, y: ((preprocess_input(x[0]), x[1]), y),
                          num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.shuffle(buffer_size=1024)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

In [ ]:
run_logdir = get_run_logdir(str(PROJECT_ROOT + "/my_logs"))

In [ ]:
checkpoint_cb = keras.callbacks.ModelCheckpoint(str(PROJECT_ROOT + "/models/model_advance_ResNetV2_TabNet.keras"),
                                                save_best_only= True,
                                                monitor="val_f1_score_macro",
                                                mode="max"
                                                )
early_stopping = keras.callbacks.EarlyStopping(patience= 20,
                                               restore_best_weights=True,
                                               monitor="val_f1_score_macro",
                                                mode="max")
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

In [ ]:
img_input = keras.layers.Input(shape=IMG_SHAPE, name="img_input_resnetv2") # (256, 256, 3)

x = layers.CenterCrop(height=224, width=224)(img_input)

base_model_resnet_v2 = ResNet50V2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model_resnet_v2.trainable = False

x = base_model_resnet_v2(x, training=False)
vision_branch = layers.GlobalAveragePooling2D()(x)

table_input = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input_tabnet")

tabnet = TabNetClassifier(n_classes=6)
tabular_branch = tabnet.tabnet_encoder(table_input)

combined = layers.concatenate([vision_branch, tabular_branch])

head = layers.Dense(256, activation="relu")(combined)
head = layers.Dropout(0.4)(head)
output = layers.Dense(6, activation="softmax", name="final_output")(head)

modelResNetV2TabNet = keras.models.Model(
    inputs=[img_input, table_input],
    outputs=[output]
)

f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')

modelResNetV2TabNet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

modelResNetV2TabNet.summary()

In [ ]:
history_ResNetV2TabNet = modelResNetV2TabNet.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    steps_per_epoch=len_train // BATCH_SIZE,
    validation_steps=len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)

# Modelo Final

Ya una vez encontrado el modelo que mejor se ajusta a los datos, se empieza con la contrucción del modelo grande, en este caso se tiene los datos de train + valid.

Epoch 29/70
4/4 ━━━━━━━━━━━━━━━━━━━━ 53s 13s/step - accuracy: 0.6724 - auc: 0.9197 - f1_score_macro: 0.7301 - loss: 0.4900 - val_accuracy: 0.7031 - val_auc: 0.8926 - val_f1_score_macro: 0.7178 - val_loss: 1.0655

In [ ]:
train_image_paths_local = [os.path.join(local_img_path, fname) for fname in df_train['img_id']]
val_image_paths_local = [os.path.join(local_img_path, fname) for fname in df_val['img_id']]

In [ ]:
full_train_paths = train_image_paths_local + val_image_paths_local
X_full_train_processed = np.concatenate([X_train_processed, X_val_processed], axis=0)
y_full_train_cat = np.concatenate([y_train_cat, y_val_cat], axis=0)

In [ ]:
print(f"Total de imágenes: {len(full_train_paths)}")
print(f"Total de filas tabulares: {X_full_train_processed.shape[0]}")
print(f"Total de etiquetas: {y_full_train_cat.shape[0]}")

In [ ]:
ds_full_train = create_fast_dataset(
    full_train_paths,
    X_full_train_processed,
    y_full_train_cat,
    augment=True  # <-- ¡La clave está aquí!
)

In [ ]:
run_logdir = get_run_logdir(str(PROJECT_ROOT + "/my_logs"))

In [ ]:
checkpoint_cb = keras.callbacks.ModelCheckpoint(str(PROJECT_ROOT + "/models/modelResNet50TabNet_pro.keras"),
                                                save_best_only= True,
                                                monitor="val_f1_score_macro",
                                                mode="max")
early_stopping = keras.callbacks.EarlyStopping(monitor="val_f1_score_macro",
                                               patience=20,
                                               restore_best_weights=True,
                                               mode="max")
tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

In [ ]:
img_input = keras.layers.Input(shape=IMG_SHAPE, name="img_input")
vision_branch = layers.CenterCrop(height=224, width=224)(img_input)

base_model = ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

vision_branch = base_model(vision_branch, training=False)
vision_branch = layers.GlobalAveragePooling2D()(vision_branch)

In [ ]:
TABULAR_SHAPE = (X_train_processed.shape[1],)

In [ ]:
table_input = keras.layers.Input(shape=TABULAR_SHAPE, name="table_input")

In [ ]:
tabnet = TabNetClassifier(n_classes= 6)
tabular_branch = tabnet.tabnet_encoder(table_input)

In [ ]:
combined = layers.concatenate([vision_branch, tabular_branch])

# Cabezal de clasificación
head = layers.Dense(256, activation="relu")(combined)
head = layers.Dropout(0.4)(head)
output = layers.Dense(6, activation="softmax", name="output")(head)

In [ ]:
model = keras.models.Model(inputs=[img_input, table_input], outputs=[output])

model.summary()

In [ ]:
f1_macro = tf.keras.metrics.F1Score(average='macro', name='f1_score_macro')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), f1_macro]
)

In [ ]:
history = model.fit(
    train_dataset,
    epochs= 40,
    validation_data=val_dataset,
    steps_per_epoch= len_train // BATCH_SIZE,
    validation_steps= len_val // BATCH_SIZE,
    callbacks=[checkpoint_cb, early_stopping, tensorboard_cb],
    class_weight=class_weights_dict
)